# RAG Data Ingestion to New Qdrant Collection\n\nUse this notebook in Google Colab to embed your markdown files and push them into a new Qdrant Cloud collection.\n\nBefore running:\n- Upload your `data/rag-data` folder as a zip, or mount Google Drive and point `DATA_DIR` to it.\n- Use a Qdrant Cloud URL, not `localhost`.\n- Set a new `QDRANT_COLLECTION_NAME` for the new database/collection.

In [ ]:
!pip -q install langchain langchain-community langchain-qdrant langchain-text-splitters qdrant-client python-dotenv

## Install and Start Ollama\n\nThis uses the same embedding model as your local project: `qwen3-embedding:4b` with dimension `2560`.

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh\nimport subprocess, time\nollama_process = subprocess.Popen(['ollama', 'serve'])\ntime.sleep(5)\n!ollama pull qwen3-embedding:4b

## Upload Data\n\nOption A: upload a zip named `rag-data.zip` that contains your markdown folder.\n\nExpected final structure can be like:\n`/content/rag-data/markdown/amazon/*.md`

In [ ]:
from google.colab import files\nuploaded = files.upload()\n\n# If you uploaded rag-data.zip, unzip it.\nimport os, zipfile\nfor name in uploaded:\n    if name.endswith('.zip'):\n        with zipfile.ZipFile(name, 'r') as z:\n            z.extractall('/content')\n        print('Extracted', name)

## Configure Qdrant\n\nUse Qdrant Cloud values here. Create a **new collection name** so this does not mix with your local partial collection.

In [ ]:
import os\n\nQDRANT_URL = 'https://YOUR-CLUSTER-URL.qdrant.io'\nQDRANT_API_KEY = 'YOUR_QDRANT_API_KEY'\nQDRANT_COLLECTION_NAME = 'financial_docs_qwen_colab_v1'\n\n# Change this if your uploaded/extracted folder path is different.\nDATA_DIR = '/content/rag-data'\nRAG_FILE_GLOB = 'markdown/**/*.md'\n\nEMBEDDING_MODEL = 'qwen3-embedding:4b'\nOLLAMA_BASE_URL = 'http://localhost:11434'\nEMBEDDING_DIMENSION = 2560\nCHUNK_SIZE = 1000\nCHUNK_OVERLAP = 200\nINGEST_BATCH_SIZE = 10\nINGEST_SLEEP_SECONDS = 0.0\n\nassert 'YOUR-CLUSTER-URL' not in QDRANT_URL, 'Set QDRANT_URL first'\nassert QDRANT_API_KEY != 'YOUR_QDRANT_API_KEY', 'Set QDRANT_API_KEY first'\nprint('Configured collection:', QDRANT_COLLECTION_NAME)

## Ingest with Resume-Safe IDs\n\nThis notebook uses deterministic point IDs. If Colab disconnects and you run it again with the same collection name and same files, existing chunks are skipped instead of duplicated.

In [ ]:
import os, re, time, uuid\nfrom langchain_community.document_loaders import DirectoryLoader, TextLoader\nfrom langchain_community.embeddings import OllamaEmbeddings\nfrom langchain_text_splitters import RecursiveCharacterTextSplitter\nfrom langchain_qdrant import QdrantVectorStore\nfrom qdrant_client import QdrantClient\nfrom qdrant_client.http.models import Distance, VectorParams\n\nPOINT_NAMESPACE = uuid.UUID('51f461c6-8f3f-45ab-bd7f-ae8f43f44629')\n\ndef enrich_metadata(documents):\n    pattern = re.compile(r'(?P<company>[a-z]+)\\s+(?P<doc_type>10-k|10-q|8-k)(?:\\s+(?P<quarter>q[1-4]))?\\s+(?P<year>\\d{4})', re.IGNORECASE)\n    for doc in documents:\n        source = doc.metadata.get('source', '')\n        filename = os.path.splitext(os.path.basename(source))[0].lower()\n        match = pattern.search(filename)\n        if not match:\n            continue\n        doc.metadata['company_name'] = match.group('company').lower()\n        doc.metadata['doc_type'] = match.group('doc_type').lower()\n        doc.metadata['fiscal_year'] = match.group('year')\n        if match.group('quarter'):\n            doc.metadata['fiscal_quarter'] = match.group('quarter').lower()\n    return documents\n\ndef point_id_for_doc(doc):\n    source = os.path.normpath(doc.metadata.get('source', '')).replace('\\\\', '/')\n    start_index = doc.metadata.get('start_index', '')\n    key = f'{source}|{start_index}|{doc.page_content[:120]}'\n    return str(uuid.uuid5(POINT_NAMESPACE, key))\n\ndef existing_ids(client, ids):\n    points = client.retrieve(\n        collection_name=QDRANT_COLLECTION_NAME,\n        ids=ids,\n        with_payload=False,\n        with_vectors=False,\n    )\n    return {str(point.id) for point in points}\n\nprint('Loading documents from', DATA_DIR)\nloader = DirectoryLoader(DATA_DIR, glob=RAG_FILE_GLOB, loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})\ndocuments = enrich_metadata(loader.load())\nprint('Loaded documents:', len(documents))\n\nsplitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP, add_start_index=True)\nsplits = splitter.split_documents(documents)\nprint('Created chunks:', len(splits))\n\nembeddings = OllamaEmbeddings(model=EMBEDDING_MODEL, base_url=OLLAMA_BASE_URL, num_ctx=1024)\nclient = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)\n\nif not client.collection_exists(QDRANT_COLLECTION_NAME):\n    print('Creating collection:', QDRANT_COLLECTION_NAME)\n    client.create_collection(\n        collection_name=QDRANT_COLLECTION_NAME,\n        vectors_config=VectorParams(size=EMBEDDING_DIMENSION, distance=Distance.COSINE),\n    )\nelse:\n    print('Collection already exists:', QDRANT_COLLECTION_NAME)\n\nvector_store = QdrantVectorStore(client=client, collection_name=QDRANT_COLLECTION_NAME, embedding=embeddings)\n\ntotal = len(splits)\ntotal_batches = (total + INGEST_BATCH_SIZE - 1) // INGEST_BATCH_SIZE\ninserted = 0\nskipped = 0\n\nfor batch_number, start in enumerate(range(0, total, INGEST_BATCH_SIZE), start=1):\n    batch = splits[start:start + INGEST_BATCH_SIZE]\n    ids = [point_id_for_doc(doc) for doc in batch]\n    present = existing_ids(client, ids)\n    missing = [(doc, point_id) for doc, point_id in zip(batch, ids) if point_id not in present]\n\n    if missing:\n        missing_docs = [doc for doc, _ in missing]\n        missing_ids = [point_id for _, point_id in missing]\n        vector_store.add_documents(documents=missing_docs, ids=missing_ids, batch_size=INGEST_BATCH_SIZE)\n        inserted += len(missing_docs)\n    skipped += len(batch) - len(missing)\n\n    done = start + len(batch)\n    pct = round((done / total) * 100, 2)\n    count = client.count(collection_name=QDRANT_COLLECTION_NAME, exact=True).count\n    print(f'Batch {batch_number}/{total_batches} | scanned {done}/{total} ({pct}%) | inserted {inserted} | skipped {skipped} | qdrant_count {count}')\n    time.sleep(INGEST_SLEEP_SECONDS)\n\nprint('Done. Inserted:', inserted, 'Skipped:', skipped)\nprint('Final Qdrant count:', client.count(collection_name=QDRANT_COLLECTION_NAME, exact=True).count)

## Use This Collection Locally\n\nAfter Colab finishes, update your local `.env` with your Qdrant Cloud values:\n\n```env\nQDRANT_URL=https://YOUR-CLUSTER-URL.qdrant.io\nQDRANT_API_KEY=YOUR_QDRANT_API_KEY\nQDRANT_COLLECTION_NAME=financial_docs_qwen_colab_v1\nEMBEDDING_MODEL=qwen3-embedding:4b\nEMBEDDING_DIMENSION=2560\n```\n\nYour retrieval code must use the same embedding model that created the collection.